# Fairness in Machine Learning Lab

**Group members:** Da and Ma

## Project objective

This project audits and mitigates fairness disparities in a binary
credit-risk classifier trained on the German Credit dataset.

The sensitive attribute is age:

- **Young:** age < 25
- **Old:** age >= 25

The prediction target is:

- **1:** good credit
- **0:** bad credit

## Fairness families examined

1. **Independence:** demographic parity
2. **Separation:** equal opportunity and equalized odds
3. **Sufficiency:** predictive parity and calibration

## Normative principle

This notebook does not search for a universally fairest model.

For every metric and mitigation, we will ask:

1. Which definition of fairness does it optimise?
2. What happens to the other fairness definitions?
3. What predictive cost is incurred?
4. Which people bear that cost?

## Reproducibility rules

Throughout the project, we will use:

- one fixed random seed;
- one fixed train/test split;
- one consistent positive-class definition;
- one consistent age-group definition;
- the same test observations for comparable models;
- age retained separately for auditing, even when removed from the
  predictive features.

Generated tables will be saved in:

`outputs/tables/`

Generated figures will be saved in:

`outputs/figures/`

Saved models and preprocessing objects will be placed in:

`outputs/models/`

In [1]:
from __future__ import annotations

import json
import platform
import sys
from pathlib import Path

import matplotlib
import numpy as np
import pandas as pd
import sklearn

In [2]:
# Find the project root robustly.
#
# This works when JupyterLab is started from:
# 1. the fairness-lab project directory, or
# 2. the notebooks directory.

CURRENT_DIRECTORY = Path.cwd().resolve()

if CURRENT_DIRECTORY.name == "notebooks":
    PROJECT_ROOT = CURRENT_DIRECTORY.parent
elif (CURRENT_DIRECTORY / "src").exists():
    PROJECT_ROOT = CURRENT_DIRECTORY
else:
    raise RuntimeError(
        "Could not locate the project root. Start JupyterLab from the "
        "fairness-lab directory or from fairness-lab/notebooks."
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Current working directory: {CURRENT_DIRECTORY}")
print(f"Project root: {PROJECT_ROOT}")

Current working directory: /home/davis/fairness-lab/notebooks
Project root: /home/davis/fairness-lab


In [3]:
from src.fairness_lab import (
    AGE_THRESHOLD,
    FIGURES_DIR,
    MODELS_DIR,
    NEGATIVE_LABEL,
    OLD_GROUP,
    OUTPUT_DIR,
    POSITIVE_LABEL,
    RANDOM_STATE,
    TABLES_DIR,
    TEST_SIZE,
    YOUNG_GROUP,
    define_age_group,
    initialise_project,
)

In [4]:
environment_info = initialise_project()

Fairness lab project initialised.
Project root: /home/davis/fairness-lab
Random seed: 42
Environment information: /home/davis/fairness-lab/outputs/environment.json


In [5]:
environment_info

{'python_version': '3.12.3 (main, Mar 23 2026, 19:04:32) [GCC 13.3.0]',
 'python_implementation': 'CPython',
 'operating_system': 'Linux-6.6.87.2-microsoft-standard-WSL2-x86_64-with-glibc2.39',
 'machine': 'x86_64',
 'random_state': 42,
 'test_size': 0.3,
 'positive_label': 1,
 'negative_label': 0,
 'age_threshold': 25,
 'young_group_definition': 'age < 25',
 'old_group_definition': 'age >= 25',
 'package_versions': {'numpy': '2.5.0',
  'pandas': '3.0.3',
  'matplotlib': '3.11.0',
  'scikit-learn': '1.9.0',
  'fairlearn': '0.14.0',
  'openml': '0.15.1',
  'jupyterlab': '4.6.0',
  'ipykernel': '7.3.0'}}

In [6]:
# Verify the central experimental configuration.

assert RANDOM_STATE == 42
assert TEST_SIZE == 0.30

assert POSITIVE_LABEL == 1
assert NEGATIVE_LABEL == 0

assert AGE_THRESHOLD == 25
assert YOUNG_GROUP == "young"
assert OLD_GROUP == "old"

assert define_age_group(18) == YOUNG_GROUP
assert define_age_group(24) == YOUNG_GROUP
assert define_age_group(25) == OLD_GROUP
assert define_age_group(40) == OLD_GROUP

print("All project configuration checks passed.")

All project configuration checks passed.


In [7]:
# Verify that the expected directories now exist.

required_directories = [
    OUTPUT_DIR,
    TABLES_DIR,
    FIGURES_DIR,
    MODELS_DIR,
    PROJECT_ROOT / "report",
    PROJECT_ROOT / "notebooks",
    PROJECT_ROOT / "src",
]

for directory in required_directories:
    assert directory.exists(), f"Missing directory: {directory}"
    print(f"Found: {directory.relative_to(PROJECT_ROOT)}")

Found: outputs
Found: outputs/tables
Found: outputs/figures
Found: outputs/models
Found: report
Found: notebooks
Found: src


In [8]:
# Display the principal software versions used in the project.

software_versions = pd.DataFrame(
    {
        "Software": [
            "Python",
            "NumPy",
            "pandas",
            "Matplotlib",
            "scikit-learn",
        ],
        "Version": [
            platform.python_version(),
            np.__version__,
            pd.__version__,
            matplotlib.__version__,
            sklearn.__version__,
        ],
    }
)

software_versions

,Software,Version
0,Python,3.12.3
1,NumPy,2.5.0
2,pandas,3.0.3
3,Matplotlib,3.11.0
4,scikit-learn,1.9.0


# Pre-computation commitments

Before examining the dataset or running the model, Da and Ma recorded
their predictions and normative preferences in `00_commitments.md`.

That file represents the group's genuine pre-analysis position and
must not be edited after the experiments begin.

Any later reflections or changes of interpretation will be documented
separately.

In [9]:
commitments_path = PROJECT_ROOT / "00_commitments.md"

if not commitments_path.exists():
    raise FileNotFoundError(
        "00_commitments.md was not found.\n"
        "Da and Ma must complete and save their pre-computation "
        "commitments before beginning Mission 1."
    )

commitments_text = commitments_path.read_text(encoding="utf-8")

print(commitments_text)

In [10]:
# Record basic file information without changing the commitment file.

commitments_information = {
    "path": str(commitments_path),
    "size_bytes": commitments_path.stat().st_size,
    "modified_timestamp": commitments_path.stat().st_mtime,
}

commitments_information

{'path': '/home/davis/fairness-lab/00_commitments.md',
 'size_bytes': 0,
 'modified_timestamp': 1782484936.3596625}

# Mission 1 — The disparity

## Question

Is the baseline classifier unfair across the two age groups, and under
which fairness definition?

## Required outputs

For each group:

- number of observations;
- base rate of good credit;
- selection rate;
- true-positive rate;
- false-positive rate;
- false-negative rate;
- positive predictive value;
- confusion-matrix counts.

For the model overall:

- accuracy;
- demographic-parity difference;
- equalized-odds difference.

The complete audit will then be repeated after age is removed from the
predictive features while being retained separately for fairness
measurement.

In [11]:
# Mission 1 status

mission_1_status = {
    "mission": 1,
    "title": "The disparity",
    "implemented": False,
    "next_step": "Load and inspect the OpenML credit-g dataset.",
}

mission_1_status

{'mission': 1,
 'title': 'The disparity',
 'implemented': False,
 'next_step': 'Load and inspect the OpenML credit-g dataset.'}

# Mission 2 — The impossibility

## Question

Can the model have equal calibration and equal false-positive rates
across groups when their base rates differ?

For each group, we will verify:

\[
\mathrm{FPR}
=
\left(\frac{p}{1-p}\right)
\left(\frac{1-\mathrm{PPV}}{\mathrm{PPV}}\right)
(1-\mathrm{FNR})
\]

where:

- \(p\) is the base rate of the positive class;
- PPV is positive predictive value;
- FNR is false-negative rate;
- FPR is false-positive rate.

The identity will be calculated from the model's own confusion-matrix
counts.

In [12]:
# Mission 2 status

mission_2_status = {
    "mission": 2,
    "title": "The impossibility",
    "implemented": False,
    "next_step": (
        "Use the Mission 1 group confusion matrices to verify "
        "Chouldechova's identity."
    ),
}

mission_2_status

{'mission': 2,
 'title': 'The impossibility',
 'implemented': False,
 'next_step': "Use the Mission 1 group confusion matrices to verify Chouldechova's identity."}

# Mission 3 — Mitigation and fairness–accuracy frontiers

The following approaches will be compared:

1. Baseline classifier
2. Reweighing
3. ExponentiatedGradient
4. ThresholdOptimizer

For each operating point, we will report:

- accuracy;
- demographic-parity difference;
- equalized-odds difference;
- group-specific TPR and FPR.

Two separate frontier figures will be produced:

1. accuracy versus demographic-parity difference;
2. accuracy versus equalized-odds difference.

The two fairness measures will not be combined into a generic
single fairness score.

In [13]:
# Mission 3 status

mission_3_status = {
    "mission": 3,
    "title": "Mitigation and fairness–accuracy frontiers",
    "implemented": False,
    "methods": [
        "Baseline",
        "Reweighing",
        "ExponentiatedGradient",
        "ThresholdOptimizer",
    ],
}

mission_3_status

{'mission': 3,
 'title': 'Mitigation and fairness–accuracy frontiers',
 'implemented': False,
 'methods': ['Baseline',
  'Reweighing',
  'ExponentiatedGradient',
  'ThresholdOptimizer']}

# Mission 4 — Chosen operating point

Da and Ma will choose exactly one operating point from the experimental
results.

The choice will not be made automatically by selecting:

- the highest accuracy;
- the smallest demographic-parity difference;
- the smallest equalized-odds difference;
- a generic combined fairness score.

The selected operating point must be defended as a normative decision.

The final argument must identify:

1. the fairness criterion being prioritised;
2. the people protected by that criterion;
3. the predictive or operational cost;
4. who bears that cost;
5. the limitations of the historical target label.

In [14]:
# Mission 4 status

mission_4_status = {
    "mission": 4,
    "title": "Chosen operating point",
    "implemented": False,
    "decision_makers": ["Da", "Ma"],
    "automatic_selection_allowed": False,
}

mission_4_status

{'mission': 4,
 'title': 'Chosen operating point',
 'implemented': False,
 'decision_makers': ['Da', 'Ma'],
 'automatic_selection_allowed': False}

# Mission 5 — Regulation and deployment limits

The chosen credit-scoring system will be examined as a potentially
deployed product.

The legal analysis will address:

- high-risk classification under the EU AI Act;
- data and bias governance;
- use of sensitive information for bias detection;
- human oversight;
- automated decision-making safeguards;
- disparate treatment and disparate impact.

The final section will identify at least:

1. two concrete legal or regulatory obligations;
2. one concrete human-oversight mechanism;
3. a reasoned deployment conclusion.

In [15]:
# Mission 5 status

mission_5_status = {
    "mission": 5,
    "title": "Regulation and deployment limits",
    "implemented": False,
    "primary_sources_required": True,
    "minimum_obligations": 2,
    "minimum_oversight_measures": 1,
}

mission_5_status

{'mission': 5,
 'title': 'Regulation and deployment limits',
 'implemented': False,
 'primary_sources_required': True,
 'minimum_obligations': 2,
 'minimum_oversight_measures': 1}

# Project status summary

This section confirms that the Step 2 project foundation is operational.

At this stage:

- the environment has been recorded;
- the random seed has been fixed;
- the output directories have been created;
- the sensitive-group definition has been tested;
- the positive-class definition has been tested;
- the pre-computation commitments have been loaded;
- the five mission sections have been prepared.

The experimental implementation begins with Mission 1.

In [16]:
project_status = pd.DataFrame(
    [
        {
            "Component": "Project directories",
            "Status": "Ready",
        },
        {
            "Component": "Environment metadata",
            "Status": "Ready",
        },
        {
            "Component": "Random seed",
            "Status": f"Fixed at {RANDOM_STATE}",
        },
        {
            "Component": "Positive label",
            "Status": f"Good credit = {POSITIVE_LABEL}",
        },
        {
            "Component": "Young group",
            "Status": f"Age < {AGE_THRESHOLD}",
        },
        {
            "Component": "Old group",
            "Status": f"Age >= {AGE_THRESHOLD}",
        },
        {
            "Component": "Commitment file",
            "Status": "Loaded",
        },
        {
            "Component": "Mission 1 analysis",
            "Status": "Not yet implemented",
        },
        {
            "Component": "Mission 2 analysis",
            "Status": "Not yet implemented",
        },
        {
            "Component": "Mission 3 analysis",
            "Status": "Not yet implemented",
        },
        {
            "Component": "Mission 4 decision",
            "Status": "Pending experimental results",
        },
        {
            "Component": "Mission 5 analysis",
            "Status": "Not yet implemented",
        },
    ]
)

project_status

,Component,Status
0,Project directories,Ready
1,Environment metadata,Ready
2,Random seed,Fixed at 42
3,Positive label,Good credit = 1
4,Young group,Age < 25
5,Old group,Age >= 25
6,Commitment file,Loaded
7,Mission 1 analysis,Not yet implemented
8,Mission 2 analysis,Not yet implemented
9,Mission 3 analysis,Not yet implemented


In [17]:
# Final Step 2 checks

assert commitments_path.exists()
assert OUTPUT_DIR.exists()
assert TABLES_DIR.exists()
assert FIGURES_DIR.exists()
assert MODELS_DIR.exists()

environment_file = OUTPUT_DIR / "environment.json"
assert environment_file.exists()

with environment_file.open("r", encoding="utf-8") as file:
    saved_environment = json.load(file)

assert saved_environment["random_state"] == RANDOM_STATE
assert saved_environment["positive_label"] == POSITIVE_LABEL
assert saved_environment["negative_label"] == NEGATIVE_LABEL
assert saved_environment["age_threshold"] == AGE_THRESHOLD

print("Step 2 completed successfully.")
print("The project is ready for Mission 1.")

Step 2 completed successfully.
The project is ready for Mission 1.
